# AR Step 3 Confounds

Raw and residual ridge-probe confound study for the AR Transformer-VAE.
The property panel and probe flow follow `transformer_vae/train-AR.ipynb`.


In [ ]:
from pathlib import Path
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

def find_project_root(start=None):
    p = (Path(start) if start is not None else Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    raise RuntimeError(f"Couldn't find project root from {p}")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from artifacts.model_compare.ar_model_h256_l256.patched_notebooks import ar_common as arc

ARTIFACT_ROOT = arc.ARTIFACT_ROOT
print("artifact_root =", ARTIFACT_ROOT)
for warning_text in arc.dependency_report():
    warnings.warn(warning_text)


In [ ]:
OUTPUT_DIR = ARTIFACT_ROOT / "confounds_step3"
TABLES_DIR = OUTPUT_DIR / "tables"
PLOTS_DIR = OUTPUT_DIR / "plots"
DIRECTIONS_DIR = OUTPUT_DIR / "confound_directions"
for path in [OUTPUT_DIR, TABLES_DIR, PLOTS_DIR, DIRECTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)


## Load Dataset And AR Checkpoint


In [ ]:
bundle = arc.load_dataset_and_splits()
model, device, ckpt_path = arc.load_ar_model(bundle)
print("device =", device)
print("checkpoint =", ckpt_path)
print("dataset rows =", len(bundle["df"]))
print("split sizes =", bundle["split_sizes"])
print("vocab size =", bundle["vocab_size"], "max_len =", bundle["max_len"])


## A. Encode Molecules Into Latent Z


In [ ]:
Z = arc.encode_latents(model, bundle["data"], device, batch_size=256)
np.save(OUTPUT_DIR / "Z_mu.npy", Z)
print("Z shape =", Z.shape)


## B-C. RDKit Property Panel And SELFIES Confounds


In [ ]:
C = arc.build_confound_panel(bundle["df"])
Y, property_warnings = arc.build_property_panel(bundle["df"]["smiles"], include_sa=True)
for text in property_warnings:
    warnings.warn(text)

panel = pd.concat(
    [
        bundle["df"][["smiles", "selfies", "split"]].reset_index(drop=True),
        C.reset_index(drop=True),
        Y.reset_index(drop=True),
    ],
    axis=1,
)
panel.to_csv(OUTPUT_DIR / "panels_step3.csv", index=False)
panel.to_csv(TABLES_DIR / "panels_step3.csv", index=False)

property_cols = [
    col for col in arc.AR_PROPERTY_COLUMNS
    if col in panel.columns and panel[col].notna().sum() > 100
]
print("properties =", property_cols)
print("confounds =", arc.CONF_COLUMNS)
display(panel.head())


## D-E. Ridge Probes And Residualization


In [ ]:
split = panel["split"].to_numpy()
C_mat = panel[arc.CONF_COLUMNS].to_numpy(dtype=float)

z_to_y_rows = []
c_to_y_rows = []
z_to_c_rows = []
z_to_yresid_rows = []

for prop in tqdm(property_cols, desc="properties"):
    y = panel[prop].to_numpy(dtype=float)

    _, _, _, _, z_scores = arc.fit_ridge_probe(Z, y, split, alpha=1e-2)
    z_to_y_rows.append({"property": prop, **z_scores})

    residual, c_model, c_scaler, c_scores = arc.residualize_against_confounds(C_mat, y, split, alpha=1e1)
    c_to_y_rows.append({"property": prop, **c_scores})
    panel[f"resid_{prop}"] = residual

    _, _, _, _, zr_scores = arc.fit_ridge_probe(Z, residual, split, alpha=1e-2)
    z_to_yresid_rows.append({
        "property": prop,
        "r2_original": z_scores["r2_val"],
        "r2_residual": zr_scores["r2_val"],
        "delta": zr_scores["r2_val"] - z_scores["r2_val"],
        "r2_original_test": z_scores["r2_test"],
        "r2_residual_test": zr_scores["r2_test"],
        "delta_test": zr_scores["r2_test"] - z_scores["r2_test"],
        "alpha_original": z_scores["alpha"],
        "alpha_residual": zr_scores["alpha"],
    })

for conf in tqdm(arc.CONF_COLUMNS, desc="confounds"):
    y = panel[conf].to_numpy(dtype=float)
    model_c, scaler_c, _, _, scores_c = arc.fit_ridge_probe(Z, y, split, alpha=1.0)
    z_to_c_rows.append({"confound": conf, **scores_c})
    w = np.asarray(model_c.coef_, dtype=float)
    w = w / max(np.linalg.norm(w), 1e-12)
    np.save(DIRECTIONS_DIR / f"w_{conf}.npy", w.astype(np.float32))

r2_Z_to_Y = pd.DataFrame(z_to_y_rows)
r2_C_to_Y = pd.DataFrame(c_to_y_rows)
r2_Z_to_C = pd.DataFrame(z_to_c_rows)
r2_Z_to_Y_vs_Yresid = pd.DataFrame(z_to_yresid_rows)

r2_Z_to_Y.to_csv(TABLES_DIR / "r2_Z_to_Y.csv", index=False)
r2_C_to_Y.to_csv(TABLES_DIR / "r2_C_to_Y.csv", index=False)
r2_Z_to_C.to_csv(TABLES_DIR / "r2_Z_to_C.csv", index=False)
r2_Z_to_Y_vs_Yresid.to_csv(TABLES_DIR / "r2_Z_to_Y_vs_Yresid.csv", index=False)

panel.to_csv(OUTPUT_DIR / "panels_step3_with_residuals.csv", index=False)
panel.to_csv(TABLES_DIR / "panels_step3_with_residuals.csv", index=False)

display(r2_Z_to_Y_vs_Yresid.sort_values("r2_residual_test", ascending=False))


## F. Correlations And Minimal Plots


In [ ]:
corr_long = arc.property_confound_correlations(panel, property_cols, arc.CONF_COLUMNS)
corr_long["abs_spearman_corr"] = corr_long["spearman_corr"].abs()
corr_long.to_csv(TABLES_DIR / "property_confound_correlations.csv", index=False)

pearson = corr_long.pivot(index="property", columns="confound", values="pearson_corr")
spearman = corr_long.pivot(index="property", columns="confound", values="spearman_corr")
pearson.to_csv(TABLES_DIR / "corr_pearson_YC.csv")
spearman.to_csv(TABLES_DIR / "corr_spearman_YC.csv")

plt.figure(figsize=(7, max(4, 0.35 * len(property_cols))))
plt.imshow(spearman.loc[property_cols, arc.CONF_COLUMNS], vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
plt.colorbar(label="Spearman correlation")
plt.xticks(range(len(arc.CONF_COLUMNS)), arc.CONF_COLUMNS, rotation=45, ha="right")
plt.yticks(range(len(property_cols)), property_cols)
plt.title("Property vs SELFIES confound correlations")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "property_confound_correlation_heatmap.png", dpi=180)
plt.show()

cmp_df = r2_Z_to_Y_vs_Yresid.sort_values("r2_original_test", ascending=True)
plt.figure(figsize=(8, max(4, 0.35 * len(cmp_df))))
y_pos = np.arange(len(cmp_df))
plt.barh(y_pos - 0.18, cmp_df["r2_original_test"], height=0.35, label="raw R^2")
plt.barh(y_pos + 0.18, cmp_df["r2_residual_test"], height=0.35, label="residual R^2")
plt.yticks(y_pos, cmp_df["property"])
plt.xlabel("test R^2")
plt.title("Raw R^2 vs residual R^2")
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "raw_vs_residual_r2.png", dpi=180)
plt.show()

zc = r2_Z_to_C.sort_values("r2_test", ascending=True)
plt.figure(figsize=(6, 3))
plt.barh(zc["confound"], zc["r2_test"])
plt.xlabel("test R^2")
plt.title("SELFIES confound predictability from Z")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "confound_predictability_from_Z.png", dpi=180)
plt.show()

drop_df = r2_Z_to_Y_vs_Yresid.assign(residualization_drop=lambda d: d["r2_residual_test"] - d["r2_original_test"])
plt.figure(figsize=(8, max(4, 0.35 * len(drop_df))))
plt.barh(drop_df["property"], drop_df["residualization_drop"])
plt.axvline(0, color="black", linewidth=1)
plt.xlabel("residual R^2 - raw R^2")
plt.title("Residualization drop")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "residualization_drop.png", dpi=180)
plt.show()

summary = {
    "artifact_root": str(OUTPUT_DIR),
    "rows": int(len(panel)),
    "split_sizes": bundle["split_sizes"],
    "properties": property_cols,
    "confounds": arc.CONF_COLUMNS,
    "avg_r2_test_z_to_y": float(r2_Z_to_Y["r2_test"].mean()),
    "avg_r2_test_c_to_y": float(r2_C_to_Y["r2_test"].mean()),
    "avg_r2_test_z_to_c": float(r2_Z_to_C["r2_test"].mean()),
    "avg_delta_r2_test_after_residualization": float(r2_Z_to_Y_vs_Yresid["delta_test"].mean()),
    "warnings": property_warnings,
}
arc.write_json(OUTPUT_DIR / "summary_step3.json", summary)
display(pd.DataFrame([summary]))
